# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Utsabsinha19/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

I will use content and search-performance fields that are available at the time of review. Numeric features will be converted to numeric values and missing numeric values will be filled with the median. Categorical features will be converted using one-hot encoding. Identifier fields such as content_id and client_id will not be included as predictive features.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update"
]

categorical_features = [
    "content_type",
    "main_intent",
    "competition_level",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier"
]

X_numeric = df[numeric_features].copy()
X_numeric = X_numeric.fillna(X_numeric.median(numeric_only=True))

X_categorical = pd.get_dummies(
    df[categorical_features],
    dummy_na=True
)

X = pd.concat([X_numeric, X_categorical], axis=1)

print("Original rows:", len(df))
print("Feature matrix shape:", X.shape)
print("Missing values remaining:", X.isna().sum().sum())

display(X.head())

Original rows: 30000
Feature matrix shape: (30000, 57)
Missing values remaining: 0


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,...,word_count_tier_1000-2000,word_count_tier_2000-3500,word_count_tier_3500+,word_count_tier_<1000,word_count_tier_nan,char_count_tier_15000-25000,char_count_tier_25000+,char_count_tier_8000-15000,char_count_tier_<8000,char_count_tier_nan
0,10.0,0.67,2.05,3221.0,20457.0,3803,29,22,17,16,...,False,True,False,False,False,True,False,False,False,False
1,90.0,0.01,0.05,2481.0,15562.0,15320,7,10,9,9,...,False,True,False,False,False,True,False,False,False,False
2,0.0,0.00,0.00,3515.0,23643.0,12581,11,14,11,11,...,False,False,True,False,False,True,False,False,False,False
3,10.0,0.00,0.00,2877.0,19116.0,11751,58,87,78,75,...,False,False,False,False,True,False,False,False,False,True
4,0.0,0.00,0.00,2803.0,17469.0,19140,24,177,145,144,...,False,True,False,False,False,True,False,False,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

Numeric features represent measurable search, traffic, engagement, content-size, age, and freshness information. Missing numeric values are filled with the median calculated from the available dataset. Categorical features describe content type, search intent, competition, age, freshness, and content-size groups and are one-hot encoded. The intended prediction point is the current review decision, so features must represent information available before that decision. Fields derived from a future outcome or from the target proxy will not be used.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total encoded features:", X.shape[1])

print("\nMissing numeric values before filling:")
print(df[numeric_features].isna().sum().sum())

print("\nMissing values after feature construction:")
print(X.isna().sum().sum())

Numeric features: 24
Categorical features: 7
Total encoded features: 57

Missing numeric values before filling:
22802

Missing values after feature construction:
0


## 3. The leakage hunt

I checked the feature set for identifiers, trend-derived fields, and fields that could overlap with a future target. content_id and client_id are identifiers rather than useful predictive signals. trend_direction and trend_pct are derived from performance changes and could overlap with a future review-priority definition, so I will exclude them from the initial feature vector. I will also document the use of 30-day and 90-day windows carefully because their measurement periods may overlap.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=== Leakage / privacy checks ===")

# Identifier check
identifier_fields = ["content_id", "client_id"]

for col in identifier_fields:
    print(
        f"{col}: present={col in df.columns}, "
        f"unique_values={df[col].nunique()}"
    )

# Trend-derived fields
trend_fields = ["trend_direction", "trend_pct"]

print("\nTrend-derived fields:")
for col in trend_fields:
    print(f"{col}: present={col in df.columns}")

# Window fields
window_fields = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

print("\nPerformance-window fields:")
for col in window_fields:
    print(f"{col}: present={col in df.columns}")

# Check whether excluded fields accidentally entered X
print("\nIdentifiers in feature matrix:")
print([col for col in identifier_fields if col in X.columns])

print("\nTrend fields in feature matrix:")
print([col for col in trend_fields if col in X.columns])

=== Leakage / privacy checks ===
content_id: present=True, unique_values=30000
client_id: present=True, unique_values=32

Trend-derived fields:
trend_direction: present=True
trend_pct: present=True

Performance-window fields:
impressions_90d: present=True
clicks_90d: present=True
sessions_90d: present=True
impressions_last_30d: present=True
clicks_last_30d: present=True
sessions_last_30d: present=True
impressions_prev_30d: present=True
clicks_prev_30d: present=True
sessions_prev_30d: present=True

Identifiers in feature matrix:
[]

Trend fields in feature matrix:
[]


## 4. What I excluded and why

content_id — excluded because it is an identifier and does not represent a generalizable content signal.

client_id — excluded because it identifies the client and could introduce client-specific patterns rather than generalizable content signals.

provider_used and model_used — excluded because they describe the data-generation/provider process rather than the content opportunity itself.

trend_direction and trend_pct — excluded from the initial feature vector because they summarize performance change and could overlap with a future target/proxy definition.

ctr, avg_position, engagement_rate, scroll_rate, and ai_traffic_pct — not included in the initial feature vector because they are outcome/performance-derived measures that could create leakage depending on how the final review-priority proxy is defined.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_fields = {
    "content_id": "Identifier; not a generalizable predictive feature.",
    "client_id": "Client identifier; may introduce client-specific patterns.",
    "provider_used": "Describes the data-generation/provider process.",
    "model_used": "Describes the data-generation/model process.",
    "trend_direction": "Performance-change summary; possible target overlap.",
    "trend_pct": "Performance-change measure; possible target overlap.",
    "ctr": "Outcome-derived performance measure; possible target leakage.",
    "avg_position": "Observed search-performance outcome; possible target leakage.",
    "engagement_rate": "Outcome-derived engagement measure; possible target leakage.",
    "scroll_rate": "Outcome-derived engagement measure; possible target leakage.",
    "ai_traffic_pct": "Observed traffic outcome; possible target leakage."
}

for field, reason in excluded_fields.items():
    print(f"{field}: {reason}")

content_id: Identifier; not a generalizable predictive feature.
client_id: Client identifier; may introduce client-specific patterns.
provider_used: Describes the data-generation/provider process.
model_used: Describes the data-generation/model process.
trend_direction: Performance-change summary; possible target overlap.
trend_pct: Performance-change measure; possible target overlap.
ctr: Outcome-derived performance measure; possible target leakage.
avg_position: Observed search-performance outcome; possible target leakage.
engagement_rate: Outcome-derived engagement measure; possible target leakage.
scroll_rate: Outcome-derived engagement measure; possible target leakage.
ai_traffic_pct: Observed traffic outcome; possible target leakage.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.